In [1]:
import random
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
import dgl
from dgl.data import AmazonCoBuyComputerDataset

In [2]:
def seed_all(s=42):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
seed_all(42)

In [3]:
DATA = Path("../data/amazon_computers.pt")
OUT = Path("../outputs/amazon_computers")
OUT.mkdir(parents=True, exist_ok=True)

In [4]:
def sample_per_class(rs, onehot, n, forbidden=None):
    n_samples, n_classes = onehot.shape
    per = {c: [] for c in range(n_classes)}
    for c in range(n_classes):
        for i in range(n_samples):
            if onehot[i, c] and (forbidden is None or i not in set(forbidden)):
                per[c].append(i)
    return np.concatenate([rs.choice(per[c], n, replace=False) for c in per])

In [5]:
def make_splits(rs, labels):
    n_classes = int(labels.max() + 1)
    N = len(labels)
    oh = np.eye(n_classes)[labels.numpy()]
    tr = sample_per_class(rs, oh, 20)
    va = sample_per_class(rs, oh, 30, forbidden=tr)
    te = np.setdiff1d(np.arange(N), np.concatenate([tr, va]))
    return torch.LongTensor(tr), torch.LongTensor(va), torch.LongTensor(te)